In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 0 – Install Dependencies (Lightning AI Ready)                 ║
# ╚══════════════════════════════════════════════════════════════════════╝

!pip install -q torch torchvision torchaudio
!pip install -q pillow numpy tqdm matplotlib scikit-image opencv-python
!pip install -q lpips

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 1 – Imports & Setup                                           ║
# ╚══════════════════════════════════════════════════════════════════════╝

import argparse
import datetime
import itertools
import time
import sys
import os
import random
from math import exp

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
from torch.utils.data import Dataset, DataLoader
from torchvision.utils import save_image
import torchvision.transforms as transforms
import torchvision
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
from torch.nn.modules.utils import _pair, _quadruple

# ── Device ────────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ── Argparse (ipykernel-safe) ─────────────────────────────────────────────────
parser = argparse.ArgumentParser()

# ── Train args ────────────────────────────────────────────────────────────────
parser.add_argument("--data_root", type=str,
                    default='/teamspace/studios/this_studio',
                    help="dataset root (trainA / trainB / trainB_label / trainC inside)")
parser.add_argument("--epoch",              type=int,   default=0)
parser.add_argument("--n_epochs",           type=int,   default=35)
parser.add_argument("--exp_name",           type=str,   default="uiess_atm_v3")
parser.add_argument("--batch_size",         type=int,   default=1)
parser.add_argument("--lr",                 type=float, default=5e-4)
parser.add_argument("--b1",                 type=float, default=0.5)
parser.add_argument("--b2",                 type=float, default=0.999)
parser.add_argument("--decay_epoch",        type=int,   default=20)
parser.add_argument("--n_cpu",              type=int,   default=0)
parser.add_argument("--img_height",         type=int,   default=128)
parser.add_argument("--img_width",          type=int,   default=128)
parser.add_argument("--channels",           type=int,   default=3)
parser.add_argument("--sample_interval",    type=int,   default=1)
parser.add_argument("--checkpoint_interval",type=int,   default=10)
parser.add_argument("--n_downsample",       type=int,   default=2)
parser.add_argument("--n_residual",         type=int,   default=3)
parser.add_argument("--dim",                type=int,   default=40)
parser.add_argument("--style_dim",          type=int,   default=8)
parser.add_argument("--gpu",                type=str,   default='0')
parser.add_argument("--seed",               type=int,   default=123)
parser.add_argument("--out_dir",            type=str,
                    default='/teamspace/studios/this_studio/output')

# ── v3 atmosphere args ────────────────────────────────────────────────────────
parser.add_argument("--lambda_atm",         type=float, default=0.5,
                    help="weight for atmospheric L1 style guidance loss")
parser.add_argument("--atm_warmup_epochs",  type=int,   default=3,
                    help="epochs to skip atmosphere loss (let base model stabilize)")

# ── Test args ─────────────────────────────────────────────────────────────────
parser.add_argument("--test_dir",  type=str,
                    default='/teamspace/studios/this_studio/testA')
parser.add_argument("--model_dir", type=str,
                    default='/teamspace/studios/this_studio/output/saved_models/uiess_atm_v3')
parser.add_argument("--checkpoint",              type=int,  default=29)
parser.add_argument("--num_sample",              type=int,  default=2)
parser.add_argument("--print_model_complexity",  type=bool, default=True)

# Jupyter / ipykernel passes extra args that argparse cannot parse
if any("ipykernel" in a or "jupyter" in a for a in sys.argv):
    opt = parser.parse_args(args=[])
else:
    opt = parser.parse_args()

print("Config:", vars(opt))

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 2 – Dataset Definitions                                       ║
# ║                                                                      ║
# ║  trainC (atmosphere clean images) loaded via deterministic modulo.   ║
# ║  Identical to original paper dataset except for trainC addition.     ║
# ╚══════════════════════════════════════════════════════════════════════╝

import os
import random

import torchvision.transforms as transforms
from PIL import Image, ImageOps
from torch.utils.data import Dataset


def is_image_file(filename):
    return any(filename.endswith(ext)
               for ext in [".bmp", ".png", ".jpg", ".jpeg", ".JPG", ".PNG"])


def load_img(filepath):
    img = Image.open(filepath).convert('RGB')
    return img


def get_patch(imgs, patch_size, scale=1, ix=-1, iy=-1):
    (ih, iw) = imgs[0].size
    tp = patch_size * scale
    ip = tp // scale
    if ix == -1:
        ix = random.randrange(0, iw - ip + 1)
    if iy == -1:
        iy = random.randrange(0, ih - ip + 1)
    tx, ty = scale * ix, scale * iy
    return tuple(img.crop((ty, tx, ty + tp, tx + tp)) for img in imgs)


def augmentation(imgs, flip_h=True, rot=True):
    info_aug = {'flip_h': False, 'flip_v': False, 'trans': False}
    if random.random() < 0.5 and flip_h:
        imgs = [ImageOps.flip(img) for img in imgs]
        info_aug['flip_h'] = True
    if rot:
        if random.random() < 0.5:
            imgs = [ImageOps.mirror(img) for img in imgs]
            info_aug['flip_v'] = True
        if random.random() < 0.5:
            imgs = [img.rotate(180) for img in imgs]
        info_aug['trans'] = True
    return tuple(imgs), info_aug


class EnhancedDataset(Dataset):
    def __init__(self, root, transforms_=None, mode="train", patch_size=128):
        super().__init__()
        self.transform  = transforms.Compose(transforms_)
        self.mode       = mode
        self.patch_size = patch_size

        if mode == 'train':
            self.filesA = sorted([os.path.join(root, 'trainA', x)
                                   for x in os.listdir(os.path.join(root, 'trainA'))
                                   if is_image_file(x)])
            self.filesB = sorted([os.path.join(root, 'trainB', x)
                                   for x in os.listdir(os.path.join(root, 'trainB'))
                                   if is_image_file(x)])
            self.labelB = sorted([os.path.join(root, 'trainB_label', x)
                                   for x in os.listdir(os.path.join(root, 'trainB_label'))
                                   if is_image_file(x)])
            # Domain C – atmospheric clean images
            trainC_path = os.path.join(root, 'trainC')
            self.filesC = (sorted([os.path.join(trainC_path, x)
                                    for x in os.listdir(trainC_path)
                                    if is_image_file(x)])
                           if os.path.isdir(trainC_path) else [])
        else:
            self.filesA = sorted([os.path.join(root, 'testA', x)
                                   for x in os.listdir(os.path.join(root, 'testA'))
                                   if is_image_file(x)])
            self.filesB = sorted([os.path.join(root, 'testB', x)
                                   for x in os.listdir(os.path.join(root, 'testB'))
                                   if is_image_file(x)])
            self.labelB = sorted([os.path.join(root, 'testB_label', x)
                                   for x in os.listdir(os.path.join(root, 'testB_label'))
                                   if is_image_file(x)])
            self.filesC = []  # no atmospheric supervision at test time

    def __getitem__(self, index):
        img_A   = load_img(self.filesA[index % len(self.filesA)])
        img_B   = load_img(self.filesB[index])
        label_B = load_img(self.labelB[index])

        if self.mode == 'train':
            img_A          = get_patch([img_A], self.patch_size)[0]
            img_B, label_B = get_patch([img_B, label_B], self.patch_size)
            (img_A, img_B, label_B), _ = augmentation([img_A, img_B, label_B])

        if self.mode == 'val':
            w, h   = img_A.size
            new_w  = w // 4 * 4 if w % 4 else w
            new_h  = h // 4 * 4 if h % 4 else h
            if new_w != w or new_h != h:
                img_A = img_A.resize((new_w, new_h))

        sample = {
            "Real":  self.transform(img_A),
            "Syn":   self.transform(img_B),
            "label": self.transform(label_B),
        }

        # Atmosphere images loaded deterministically via modulo
        if self.filesC:
            img_C = load_img(self.filesC[index % len(self.filesC)])
            if self.mode == 'train':
                img_C      = get_patch([img_C], self.patch_size)[0]
                (img_C,), _ = augmentation([img_C])
            sample["Atm"] = self.transform(img_C)

        return sample

    def __len__(self):
        return len(self.filesA) if self.mode == 'val' else len(self.filesB)


class EnhancedValDataset(Dataset):
    def __init__(self, transforms_=None, dataset_path="train", patch_size=128):
        super().__init__()
        self.transform = transforms.Compose(transforms_)
        self.files     = [os.path.join(dataset_path, x)
                          for x in os.listdir(dataset_path)
                          if is_image_file(x)]

    def __getitem__(self, index):
        img = load_img(self.files[index % len(self.files)])
        w, h  = img.size
        new_w = w // 4 * 4 if w % 4 else w
        new_h = h // 4 * 4 if h % 4 else h
        if new_w != w or new_h != h:
            img = img.resize((new_w, new_h))
        img = self.transform(img)
        return {"img": img, "name": self.files[index % len(self.files)]}

    def __len__(self):
        return len(self.files)

print(f"Dataset classes defined. trainC available: {os.path.isdir(os.path.join(opt.data_root, 'trainC'))}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 3 – Model Definitions (IDENTICAL to paper)                    ║
# ╚══════════════════════════════════════════════════════════════════════╝

def weights_init_normal(m):
    classname = m.__class__.__name__
    if classname.find("Conv2d") != -1:
        torch.nn.init.kaiming_normal_(m.weight)
        if m.bias is not None:
            m.bias.data.zero_()
    elif classname.find("BatchNorm2d") != -1:
        torch.nn.init.kaiming_normal_(m.weight)
        if m.bias is not None:
            m.bias.data.zero_()


class LambdaLR:
    def __init__(self, n_epochs, offset, decay_start_epoch):
        assert (n_epochs - decay_start_epoch) > 0, \
            "Decay must start before the training session ends!"
        self.n_epochs = n_epochs
        self.offset = offset
        self.decay_start_epoch = decay_start_epoch

    def step(self, epoch):
        return 1.0 - max(0, epoch + self.offset - self.decay_start_epoch) / (
            self.n_epochs - self.decay_start_epoch)


# ─── Custom Layers ────────────────────────────────────────────────────────────

class AdaptiveInstanceNorm2d(nn.Module):
    def __init__(self, num_features, eps=1e-5, momentum=0.1):
        super(AdaptiveInstanceNorm2d, self).__init__()
        self.num_features = num_features
        self.eps = eps
        self.momentum = momentum
        self.weight = None
        self.bias   = None
        self.register_buffer("running_mean", torch.zeros(num_features))
        self.register_buffer("running_var",  torch.ones(num_features))

    def forward(self, x):
        assert self.weight is not None and self.bias is not None, \
            "Please assign weight and bias before calling AdaIN!"
        b, c, h, w = x.size()
        running_mean = self.running_mean.repeat(b)
        running_var  = self.running_var.repeat(b)
        x_reshaped   = x.contiguous().view(1, b * c, h, w)
        out = F.batch_norm(x_reshaped, running_mean, running_var,
                           self.weight, self.bias, True,
                           self.momentum, self.eps)
        return out.view(b, c, h, w)

    def __repr__(self):
        return self.__class__.__name__ + "(" + str(self.num_features) + ")"


class LayerNorm(nn.Module):
    def __init__(self, num_features, eps=1e-5, affine=True):
        super(LayerNorm, self).__init__()
        self.num_features = num_features
        self.affine = affine
        self.eps    = eps
        if self.affine:
            self.gamma = nn.Parameter(torch.Tensor(num_features).uniform_())
            self.beta  = nn.Parameter(torch.zeros(num_features))

    def forward(self, x):
        shape = [-1] + [1] * (x.dim() - 1)
        mean  = x.view(x.size(0), -1).mean(1).view(*shape)
        std   = x.view(x.size(0), -1).std(1).view(*shape)
        x     = (x - mean) / (std + self.eps)
        if self.affine:
            shape = [1, -1] + [1] * (x.dim() - 2)
            x = x * self.gamma.view(*shape) + self.beta.view(*shape)
        return x


# ─── Custom Blocks ────────────────────────────────────────────────────────────

class ResidualBlock(nn.Module):
    def __init__(self, features, norm="in"):
        super(ResidualBlock, self).__init__()
        norm_layer = AdaptiveInstanceNorm2d if norm == "adain" else nn.InstanceNorm2d
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(features, features, 3),
            norm_layer(features),
            nn.ReLU(inplace=True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(features, features, 3),
            norm_layer(features),
        )

    def forward(self, x):
        return x + self.block(x)


# ─── Encoders ─────────────────────────────────────────────────────────────────

class ContentEncoder(nn.Module):
    def __init__(self, in_channels=3, dim=64, n_residual=3, n_downsample=2):
        super(ContentEncoder, self).__init__()
        layers = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(in_channels, dim, 7),
            nn.InstanceNorm2d(dim),
            nn.ReLU(inplace=True),
        ]
        for _ in range(n_downsample):
            layers += [
                nn.Conv2d(dim, dim * 2, 4, stride=2, padding=1),
                nn.InstanceNorm2d(dim * 2),
                nn.ReLU(inplace=True),
            ]
            dim *= 2
        for _ in range(n_residual):
            layers += [ResidualBlock(dim, norm="in")]
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)


class StyleEncoder(nn.Module):
    def __init__(self, in_channels=3, dim=64, n_downsample=2, style_dim=8):
        super(StyleEncoder, self).__init__()
        layers = [nn.ReflectionPad2d(3), nn.Conv2d(in_channels, dim, 7), nn.ReLU(inplace=True)]
        for _ in range(2):
            layers += [nn.Conv2d(dim, dim * 2, 4, stride=2, padding=1), nn.ReLU(inplace=True)]
            dim *= 2
        for _ in range(n_downsample - 2):
            layers += [nn.Conv2d(dim, dim, 4, stride=2, padding=1), nn.ReLU(inplace=True)]
        layers += [nn.AdaptiveAvgPool2d(1), nn.Conv2d(dim, style_dim, 1, 1, 0)]
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)


# ─── MLP ──────────────────────────────────────────────────────────────────────

class MLP(nn.Module):
    def __init__(self, input_dim, output_dim, dim=256, n_blk=3, activ="relu"):
        super(MLP, self).__init__()
        layers = [nn.Linear(input_dim, dim), nn.ReLU(inplace=True)]
        for _ in range(n_blk - 2):
            layers += [nn.Linear(dim, dim), nn.ReLU(inplace=True)]
        layers += [nn.Linear(dim, output_dim)]
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x.view(x.size(0), -1))


# ─── Generator ────────────────────────────────────────────────────────────────

class Generator(nn.Module):
    def __init__(self, out_channels=3, dim=64, n_residual=3, n_upsample=2, style_dim=8):
        super(Generator, self).__init__()
        layers = []
        dim = dim * 2 ** n_upsample
        for _ in range(n_residual):
            layers += [ResidualBlock(dim, norm="adain")]
        for _ in range(n_upsample):
            layers += [
                nn.Upsample(scale_factor=2),
                nn.Conv2d(dim, dim // 2, 5, stride=1, padding=2),
                LayerNorm(dim // 2),
                nn.ReLU(inplace=True),
            ]
            dim = dim // 2
        layers += [nn.ReflectionPad2d(3), nn.Conv2d(dim, out_channels, 7), nn.Tanh()]
        self.model = nn.Sequential(*layers)
        num_adain_params = self.get_num_adain_params()
        self.mlp = MLP(style_dim, num_adain_params)

    def get_num_adain_params(self):
        num_adain_params = 0
        for m in self.modules():
            if m.__class__.__name__ == "AdaptiveInstanceNorm2d":
                num_adain_params += 2 * m.num_features
        return num_adain_params

    def assign_adain_params(self, adain_params):
        for m in self.modules():
            if m.__class__.__name__ == "AdaptiveInstanceNorm2d":
                mean = adain_params[:, : m.num_features]
                std  = adain_params[:, m.num_features: 2 * m.num_features]
                m.bias   = mean.contiguous().view(-1)
                m.weight = std.contiguous().view(-1)
                if adain_params.size(1) > 2 * m.num_features:
                    adain_params = adain_params[:, 2 * m.num_features:]

    def forward(self, content_code, style_code):
        self.assign_adain_params(self.mlp(style_code))
        img = self.model(content_code)
        return img


# ─── Style Transform Unit ─────────────────────────────────────────────────────

class StyleTransformUnit(nn.Module):
    def __init__(self, dim=64, style_dim=8):
        super(StyleTransformUnit, self).__init__()
        self.estimator = nn.Sequential(
            nn.Flatten(),
            nn.Linear(style_dim, dim),
            nn.PReLU(),
            nn.Linear(dim, style_dim),
        )

    def forward(self, style_code):
        new_style_code = style_code + self.estimator(style_code).view(-1, 1, 1)
        return new_style_code


# ─── Multi-scale Discriminator ────────────────────────────────────────────────

class MultiDiscriminator(nn.Module):
    def __init__(self, in_channels=3):
        super(MultiDiscriminator, self).__init__()

        def discriminator_block(in_filters, out_filters, normalize=True):
            layers = [nn.Conv2d(in_filters, out_filters, 4, stride=2, padding=1)]
            if normalize:
                layers.append(nn.InstanceNorm2d(out_filters))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers

        self.models = nn.ModuleList()
        for i in range(3):
            self.models.add_module(
                "disc_%d" % i,
                nn.Sequential(
                    *discriminator_block(in_channels, 64, normalize=False),
                    *discriminator_block(64, 128),
                    *discriminator_block(128, 256),
                    *discriminator_block(256, 512),
                    nn.Conv2d(512, 1, 3, padding=1)
                ),
            )
        self.downsample = nn.AvgPool2d(in_channels, stride=2, padding=[1, 1],
                                       count_include_pad=False)

    def compute_loss(self, x, gt):
        loss = sum([torch.mean((out - gt) ** 2) for out in self.forward(x)])
        return loss

    def forward(self, x):
        outputs = []
        for m in self.models:
            outputs.append(m(x))
            x = self.downsample(x)
        return outputs


print("Model classes defined.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 4 – Loss Functions (IDENTICAL to paper)                       ║
# ╚══════════════════════════════════════════════════════════════════════╝

def gaussian(window_size, sigma):
    gauss = torch.Tensor([
        exp(-(x - window_size // 2) ** 2 / float(2 * sigma ** 2))
        for x in range(window_size)
    ])
    return gauss / gauss.sum()


def create_window(window_size, channel=1):
    _1D_window = gaussian(window_size, 1.5).unsqueeze(1)
    _2D_window = _1D_window.mm(_1D_window.t()).float().unsqueeze(0).unsqueeze(0)
    window = _2D_window.expand(channel, 1, window_size, window_size).contiguous()
    return window


def ssim(img1, img2, window_size=11, window=None,
         size_average=True, full=False, val_range=None):
    if val_range is None:
        max_val = 255 if torch.max(img1) > 128 else 1
        min_val = -1  if torch.min(img1) < -0.5 else 0
        L = max_val - min_val
    else:
        L = val_range

    padd = 0
    (_, channel, height, width) = img1.size()
    if window is None:
        real_size = min(window_size, height, width)
        window    = create_window(real_size, channel=channel).to(img1.device)

    mu1 = F.conv2d(img1, window, padding=padd, groups=channel)
    mu2 = F.conv2d(img2, window, padding=padd, groups=channel)

    mu1_sq  = mu1.pow(2)
    mu2_sq  = mu2.pow(2)
    mu1_mu2 = mu1 * mu2

    sigma1_sq = F.conv2d(img1 * img1, window, padding=padd, groups=channel) - mu1_sq
    sigma2_sq = F.conv2d(img2 * img2, window, padding=padd, groups=channel) - mu2_sq
    sigma12   = F.conv2d(img1 * img2, window, padding=padd, groups=channel) - mu1_mu2

    C1 = (0.01 * L) ** 2
    C2 = (0.03 * L) ** 2

    v1 = 2.0 * sigma12 + C2
    v2 = sigma1_sq + sigma2_sq + C2
    cs = torch.mean(v1 / v2)

    ssim_map = ((2 * mu1_mu2 + C1) * v1) / ((mu1_sq + mu2_sq + C1) * v2)

    if size_average:
        ret = ssim_map.mean()
    else:
        ret = ssim_map.mean(1).mean(1).mean(1)

    if full:
        return ret, cs
    return ret


class SSIM(torch.nn.Module):
    def __init__(self, window_size=11, size_average=True, val_range=None):
        super(SSIM, self).__init__()
        self.window_size  = window_size
        self.size_average = size_average
        self.val_range    = val_range
        self.channel = 1
        self.window  = create_window(window_size)

    def forward(self, img1, img2):
        (_, channel, _, _) = img1.size()
        if channel == self.channel and self.window.dtype == img1.dtype:
            window = self.window
        else:
            window = create_window(self.window_size, channel).to(img1.device).type(img1.dtype)
            self.window  = window
            self.channel = channel
        return ssim(img1, img2, window=window, window_size=self.window_size,
                    size_average=self.size_average, val_range=1)


class TVLoss(nn.Module):
    def __init__(self, TVLoss_weight=1):
        super(TVLoss, self).__init__()
        self.TVLoss_weight = TVLoss_weight

    def forward(self, x):
        batch_size = x.size()[0]
        h_x = x.size()[2]
        w_x = x.size()[3]
        count_h = self._tensor_size(x[:, :, 1:, :])
        count_w = self._tensor_size(x[:, :, :, 1:])
        h_tv = torch.pow((x[:, :, 1:, :] - x[:, :, :h_x - 1, :]), 2).sum()
        w_tv = torch.pow((x[:, :, :, 1:] - x[:, :, :, :w_x - 1]), 2).sum()
        return self.TVLoss_weight * 2 * (h_tv / count_h + w_tv / count_w) / batch_size

    def _tensor_size(self, t):
        return t.size()[1] * t.size()[2] * t.size()[3]


class VGGPerceptualLoss(torch.nn.Module):
    def __init__(self, weighting=1, resize=False):
        super(VGGPerceptualLoss, self).__init__()
        blocks = []
        blocks.append(torchvision.models.vgg16(pretrained=True).features[:4].eval())
        blocks.append(torchvision.models.vgg16(pretrained=True).features[4:9].eval())
        blocks.append(torchvision.models.vgg16(pretrained=True).features[9:16].eval())
        blocks.append(torchvision.models.vgg16(pretrained=True).features[16:23].eval())
        for bl in blocks:
            for p in bl:
                p.requires_grad = False
        self.blocks    = torch.nn.ModuleList(blocks)
        self.transform = torch.nn.functional.interpolate
        self.mean      = torch.nn.Parameter(
            torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.std       = torch.nn.Parameter(
            torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        self.resize    = resize
        self.weighting = weighting

    def forward(self, input, target, feature_layers=[0, 1, 2], style_layers=[2, 3]):
        if input.shape[1] != 3:
            input  = input.repeat(1, 3, 1, 1)
            target = target.repeat(1, 3, 1, 1)
        input  = (input  - self.mean) / self.std
        target = (target - self.mean) / self.std
        if self.resize:
            input  = self.transform(input,  mode='bilinear', size=(224, 224), align_corners=False)
            target = self.transform(target, mode='bilinear', size=(224, 224), align_corners=False)
        loss = 0.0
        x, y = input, target
        for i, block in enumerate(self.blocks):
            x = block(x)
            y = block(y)
            if i in feature_layers:
                loss += torch.nn.functional.l1_loss(x, y)
            if i in style_layers:
                act_x = x.reshape(x.shape[0], x.shape[1], -1)
                act_y = y.reshape(y.shape[0], y.shape[1], -1)
                gram_x = act_x @ act_x.permute(0, 2, 1)
                gram_y = act_y @ act_y.permute(0, 2, 1)
                loss += torch.nn.functional.l1_loss(gram_x, gram_y)
        return self.weighting * loss


print("Loss functions defined.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 5 – Training Function  (v3 — fixed)                              ║
# ║                                                                          ║
# ║  Fixes from v2 → v3:                                                   ║
# ║   F1  L1 atmosphere loss instead of cosine similarity                   ║
# ║       → both direction AND magnitude matter for AdaIN parameters       ║
# ║   F2  atm_sty_Enc added to optimizer_G (no separate optimizer)          ║
# ║       → gradient flows: loss_atm → en_s_code_B → T                    ║
# ║   F3  NO gradient clipping (matches original paper)                    ║
# ║   F4  Guide ONLY en_s_code_B (not both A and B)                        ║
# ║       → paper's Llatent already constrains en_s_code_B ≈ en_s_code_A  ║
# ║   F5  Per-epoch PSNR/SSIM validation + best model tracking             ║
# ║   F6  Warmup = 3 epochs (let base model form latent space first)       ║
# ║                                                                          ║
# ║  Paper parameters UNCHANGED:                                             ║
# ║   All lambda values, LR, betas, batch size, architecture               ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import os, sys, time, random, datetime, itertools
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision.utils import save_image
import torchvision.transforms as transforms
from PIL import Image


def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark     = False
        torch.backends.cudnn.deterministic = True


def worker_init(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)


def train():
    images_dir = os.path.join(opt.out_dir, "images",       opt.exp_name)
    models_dir = os.path.join(opt.out_dir, "saved_models", opt.exp_name)
    os.makedirs(images_dir, exist_ok=True)
    os.makedirs(models_dir, exist_ok=True)

    set_seed(opt.seed)

    # ── Loss functions (identical to paper) ───────────────────────────────────
    criterion_recon = torch.nn.L1Loss().to(device)
    ssim_loss       = SSIM().to(device)
    tv_loss         = TVLoss(1).to(device)
    perceptual_loss = VGGPerceptualLoss().to(device)

    # ── Models (identical to paper) ───────────────────────────────────────────
    c_Enc        = ContentEncoder(dim=opt.dim, n_downsample=opt.n_downsample,
                                  n_residual=opt.n_residual).to(device)
    G            = Generator(dim=opt.dim, n_upsample=opt.n_downsample,
                             n_residual=opt.n_residual,
                             style_dim=opt.style_dim).to(device)
    real_sty_Enc = StyleEncoder(dim=opt.dim, n_downsample=opt.n_downsample,
                                style_dim=opt.style_dim).to(device)
    syn_sty_Enc  = StyleEncoder(dim=opt.dim, n_downsample=opt.n_downsample,
                                style_dim=opt.style_dim).to(device)
    T            = StyleTransformUnit(dim=opt.dim,
                                      style_dim=opt.style_dim).to(device)
    D            = MultiDiscriminator().to(device)

    # ── [NEW] Atmospheric style encoder ───────────────────────────────────────
    # Same architecture as syn_sty_Enc. Included in optimizer_G (FIX F2).
    atm_sty_Enc = StyleEncoder(dim=opt.dim, n_downsample=opt.n_downsample,
                               style_dim=opt.style_dim).to(device)

    # ── Checkpoint loading ────────────────────────────────────────────────────
    if opt.epoch != 0:
        def _load(name, model):
            path = os.path.join(models_dir, f"{name}_{opt.epoch}.pth")
            model.load_state_dict(torch.load(path, map_location=device))

        for name, model in [("c_Enc", c_Enc), ("G", G),
                             ("real_sty_Enc", real_sty_Enc),
                             ("syn_sty_Enc", syn_sty_Enc),
                             ("T", T), ("D", D)]:
            _load(name, model)

        atm_ckpt = os.path.join(models_dir, f"atm_sty_Enc_{opt.epoch}.pth")
        if os.path.isfile(atm_ckpt):
            atm_sty_Enc.load_state_dict(torch.load(atm_ckpt, map_location=device))
            print(f"  Resumed atm_sty_Enc from epoch {opt.epoch}")
        else:
            atm_sty_Enc.apply(weights_init_normal)
            print("  atm_sty_Enc checkpoint not found — fresh init.")
    else:
        for m in [c_Enc, G, real_sty_Enc, syn_sty_Enc, T, D, atm_sty_Enc]:
            m.apply(weights_init_normal)

    # ── Loss weights — EXACT paper values (Section IV-A) ─────────────────────
    lambda_gan             = 1
    lambda_id              = 10
    lambda_cyc             = 1
    lambda_enhanced        = 3.5 / 2
    lambda_ssim            = 5.0 / 2
    lambda_tv              = 0.3
    lambda_perceptual      = 0.0005 / 2
    lambda_enhanced_latent = 3

    # ── [NEW] Atmospheric guidance weight ─────────────────────────────────────
    lambda_atm        = opt.lambda_atm           # default 0.5
    atm_warmup_epochs = opt.atm_warmup_epochs    # default 3

    # ── Optimizers ────────────────────────────────────────────────────────────
    # FIX F2: atm_sty_Enc is IN optimizer_G (not separate)
    # This lets the atmosphere loss gradient flow through:
    #   loss_atm → en_s_code_B → T (and also through atm_sty_Enc)
    optimizer_G = torch.optim.Adam(
        itertools.chain(c_Enc.parameters(), G.parameters(),
                        real_sty_Enc.parameters(), syn_sty_Enc.parameters(),
                        T.parameters(), atm_sty_Enc.parameters()),
        lr=opt.lr, betas=(opt.b1, opt.b2))

    optimizer_D1 = torch.optim.Adam(
        D.parameters(), lr=opt.lr * 5, betas=(opt.b1, opt.b2))

    # ── LR schedulers ─────────────────────────────────────────────────────────
    lr_scheduler_G = torch.optim.lr_scheduler.LambdaLR(
        optimizer_G,
        lr_lambda=LambdaLR(opt.n_epochs, opt.epoch, opt.decay_epoch).step)
    lr_scheduler_D1 = torch.optim.lr_scheduler.LambdaLR(
        optimizer_D1,
        lr_lambda=LambdaLR(opt.n_epochs, opt.epoch, opt.decay_epoch).step)

    # ── Data loaders ──────────────────────────────────────────────────────────
    transforms_train = [transforms.ToTensor()]
    transforms_val   = [
        transforms.Resize(size=(opt.img_height * 2, opt.img_width * 2),
                          interpolation=Image.BICUBIC),
        transforms.ToTensor(),
    ]

    set_seed(opt.seed)
    dataloader = DataLoader(
        EnhancedDataset(opt.data_root, transforms_=transforms_train),
        batch_size=opt.batch_size, shuffle=True,
        num_workers=0, worker_init_fn=worker_init, pin_memory=True)

    set_seed(opt.seed)
    val_dataloader = DataLoader(
        EnhancedDataset(opt.data_root, transforms_=transforms_val, mode="val"),
        batch_size=5, shuffle=True,
        num_workers=0, worker_init_fn=worker_init, pin_memory=True)

    # ── FIX F5: Validation data for PSNR/SSIM tracking ───────────────────────
    # Use testB + testB_label for per-epoch quantitative validation
    testB_path = os.path.join(opt.data_root, 'testB')
    testB_label_path = os.path.join(opt.data_root, 'testB_label')
    has_val_data = os.path.isdir(testB_path) and os.path.isdir(testB_label_path)
    best_psnr = 0.0
    best_epoch = 0
    val_history = []

    # ── Sample helper (identical to paper) ────────────────────────────────────
    def sample_images(batches_done):
        imgs = next(iter(val_dataloader))
        img_enhanceds_A = img_enhanceds_B = None

        for imgA, imgB, label_B in zip(imgs["Real"], imgs["Syn"], imgs["label"]):
            with torch.no_grad():
                XA = imgA.unsqueeze(0).to(device)
                XB = imgB.unsqueeze(0).to(device)

                c_code_A, s_code_A = c_Enc(XA), real_sty_Enc(XA)
                c_code_B, s_code_B = c_Enc(XB), syn_sty_Enc(XB)

                XAB = G(c_code_A, s_code_B)
                XBA = G(c_code_B, s_code_A)

                en_s_code_A = T(s_code_A)
                en_s_code_B = T(s_code_B)

                enhanced_A = G(c_code_A, en_s_code_A)
                enhanced_B = G(c_code_B, en_s_code_B)

                c_code_BA, _ = c_Enc(XBA), real_sty_Enc(XBA)
                c_code_AB, _ = c_Enc(XAB), syn_sty_Enc(XAB)
                XABA = G(c_code_AB, s_code_A) if lambda_cyc > 0 else 0
                XBAB = G(c_code_BA, s_code_B) if lambda_cyc > 0 else 0

                XAA = G(c_code_A, s_code_A)
                XBB = G(c_code_B, s_code_B)

                lb  = label_B.unsqueeze(0).to(device)
                row_B = imgB.unsqueeze(0).to(device)
                for item in [XBB, XBA, XBAB, enhanced_B, lb]:
                    row_B = torch.cat((row_B, item), -1)

                row_A = imgA.unsqueeze(0).to(device)
                for item in [XAA, XAB, XABA, enhanced_A]:
                    row_A = torch.cat((row_A, item), -1)

                img_enhanceds_A = row_A if img_enhanceds_A is None else torch.cat((img_enhanceds_A, row_A), -2)
                img_enhanceds_B = row_B if img_enhanceds_B is None else torch.cat((img_enhanceds_B, row_B), -2)

        save_image(img_enhanceds_A,
                   os.path.join(images_dir, f"{batches_done}_I2I_Enhanced_A.png"),
                   nrow=5, normalize=True, value_range=(0, 1))
        save_image(img_enhanceds_B,
                   os.path.join(images_dir, f"{batches_done}_I2I_Enhanced_B.png"),
                   nrow=5, normalize=True, value_range=(0, 1))

    # ── FIX F5: per-epoch validation ──────────────────────────────────────────
    def validate_epoch(epoch_num):
        """Compute PSNR and SSIM on testB vs testB_label using syn_sty_Enc path."""
        if not has_val_data:
            return 0.0, 0.0

        c_Enc.eval(); G.eval(); syn_sty_Enc.eval(); T.eval()

        val_ds = EnhancedValDataset(
            transforms_=[transforms.ToTensor()],
            dataset_path=testB_path)
        val_dl = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=0)

        gt_files = sorted([f for f in os.listdir(testB_label_path) if is_image_file(f)])

        psnr_sum = 0.0
        ssim_sum = 0.0
        count = 0

        for i, batch in enumerate(val_dl):
            if i >= len(gt_files):
                break
            img_syn = batch["img"].to(device)
            with torch.no_grad():
                c_code = c_Enc(img_syn)
                s_code = syn_sty_Enc(img_syn)
                en_s   = T(s_code)
                enhanced = G(c_code, en_s)

            # Load GT
            gt_path = os.path.join(testB_label_path, gt_files[i])
            gt_img = load_img(gt_path)
            # Resize enhanced to GT size
            pred_np = enhanced.squeeze().mul(255).add_(0.5).clamp_(0, 255).permute(1, 2, 0).cpu().numpy().astype(np.uint8)
            pred_pil = Image.fromarray(pred_np)
            pred_pil = pred_pil.resize(gt_img.size)
            pred_np = np.array(pred_pil).astype(np.float64)
            gt_np   = np.array(gt_img).astype(np.float64)

            # PSNR
            mse = np.mean((pred_np - gt_np) ** 2)
            if mse == 0:
                psnr = 100.0
            else:
                psnr = 10.0 * np.log10(255.0 ** 2 / mse)
            psnr_sum += psnr

            # SSIM (simple per-channel)
            from skimage.metrics import structural_similarity
            s = structural_similarity(gt_np, pred_np, channel_axis=2, data_range=255)
            ssim_sum += s
            count += 1

        c_Enc.train(); G.train(); syn_sty_Enc.train(); T.train()

        avg_psnr = psnr_sum / max(count, 1)
        avg_ssim = ssim_sum / max(count, 1)
        return avg_psnr, avg_ssim

    # ── Training loop ─────────────────────────────────────────────────────────
    valid     = 1
    fake      = 0
    prev_time = time.time()
    sample_images(0)

    for epoch in range(opt.epoch + 1 if opt.epoch > 0 else 0, opt.n_epochs + 1):
        for i, batch in enumerate(dataloader):

            # ── Inputs ───────────────────────────────────────────────────────
            XA     = batch["Real"].to(device)
            XB     = batch["Syn"].to(device)
            labelB = batch["label"].to(device)

            has_atm   = "Atm" in batch
            apply_atm = has_atm and (epoch >= atm_warmup_epochs)
            if has_atm:
                XC = batch["Atm"].to(device)

            # ── Paper forward pass (UNCHANGED) ────────────────────────────────
            c_code_A, s_code_A = c_Enc(XA), real_sty_Enc(XA)
            c_code_B, s_code_B = c_Enc(XB), syn_sty_Enc(XB)

            XAA = G(c_code_A, s_code_A)
            XBB = G(c_code_B, s_code_B)
            XBA = G(c_code_B, s_code_A)
            XAB = G(c_code_A, s_code_B)

            c_code_BA, s_code_BA = c_Enc(XBA), real_sty_Enc(XBA)
            c_code_AB, s_code_AB = c_Enc(XAB), syn_sty_Enc(XAB)
            XABA = G(c_code_AB, s_code_A) if lambda_cyc > 0 else 0
            XBAB = G(c_code_BA, s_code_B) if lambda_cyc > 0 else 0

            en_s_code_A = T(s_code_A)   # Z_S_R→C
            en_s_code_B = T(s_code_B)   # Z_S_S→C

            en_A = G(c_code_B, en_s_code_A)
            en_B = G(c_code_B, en_s_code_B)

            # ── Discriminator (IDENTICAL to paper) ────────────────────────────
            optimizer_D1.zero_grad()
            loss_D1 = (D.compute_loss(XA,          valid)
                     + D.compute_loss(XBA.detach(), fake)
                     + D.compute_loss(XB,          valid)
                     + D.compute_loss(XAB.detach(), fake))
            loss_D1.backward()
            # FIX F3: NO gradient clipping (matches original paper)
            optimizer_D1.step()

            # ── Generator ────────────────────────────────────────────────────
            optimizer_G.zero_grad()

            # ── Paper losses (ALL UNCHANGED) ─────────────────────────────────
            loss_GAN_1           = lambda_gan * (
                D.compute_loss(XBA, valid) + D.compute_loss(XAB, valid))
            loss_ID_1            = lambda_id  * criterion_recon(XAA, XA)
            loss_ID_2            = lambda_id  * criterion_recon(XBB, XB)
            loss_cyc_1           = lambda_cyc * criterion_recon(XABA, XA)
            loss_cyc_2           = lambda_cyc * criterion_recon(XBAB, XB)
            loss_enhanced        = lambda_enhanced * (
                criterion_recon(en_A, labelB) + criterion_recon(en_B, labelB))
            loss_ssim            = lambda_ssim * (
                (1 - ssim_loss(en_A, labelB)) + (1 - ssim_loss(en_B, labelB)))
            loss_perceptual      = lambda_perceptual * (
                perceptual_loss(en_A, labelB) + perceptual_loss(en_B, labelB))
            loss_enhanced_latent = lambda_enhanced_latent * criterion_recon(
                en_s_code_B, en_s_code_A)
            loss_tv              = lambda_tv * (tv_loss(en_B) + tv_loss(en_A))

            # ── FIX F1+F4: L1 atmosphere loss, ONLY en_s_code_B ──────────────
            #
            # FIX F1: L1 instead of cosine similarity.
            #   Style codes are 8-dim vectors mapped to AdaIN params.
            #   Both direction AND magnitude matter for brightness/contrast.
            #   Cosine only matches direction → magnitude mismatch → blurry.
            #   L1 matches the full vector — same loss type as Llatent.
            #
            # FIX F4: Guide ONLY en_s_code_B (syn→clean).
            #   The paper's Llatent already enforces en_s_code_B ≈ en_s_code_A.
            #   Guiding B automatically guides A via Llatent.
            #   Guiding both doubles the atm signal and can overwhelm.
            #
            # s_code_C is DETACHED: atm_sty_Enc gets gradient from optimizer_G
            # but the atmosphere loss drives T to produce en_s_code_B ≈ s_code_C,
            # not the other way around.
            if apply_atm:
                s_code_C = atm_sty_Enc(XC)
                loss_atm = lambda_atm * criterion_recon(
                    en_s_code_B, s_code_C.detach())
            else:
                loss_atm = torch.tensor(0.0, device=device)

            # ── Total loss ───────────────────────────────────────────────────
            loss_G = (loss_GAN_1 + loss_ID_1 + loss_ID_2
                    + loss_cyc_1 + loss_cyc_2
                    + loss_enhanced + loss_ssim + loss_enhanced_latent
                    + loss_perceptual + loss_tv
                    + loss_atm)
            loss_G.backward()
            # FIX F3: NO gradient clipping
            optimizer_G.step()

            # ── Logging ───────────────────────────────────────────────────────
            batches_done = epoch * len(dataloader) + i
            batches_left = opt.n_epochs * len(dataloader) - batches_done
            time_left    = datetime.timedelta(
                seconds=batches_left * (time.time() - prev_time))
            prev_time = time.time()

            if i % 1000 == 0:
                E = (loss_enhanced.item() + loss_ssim.item()
                   + loss_enhanced_latent.item() + loss_perceptual.item()
                   + loss_tv.item())
                atm_tag = f"{loss_atm.item():.4f}" if apply_atm else "warmup"
                print(
                    f"[Epoch {epoch}/{opt.n_epochs}] [Batch {i}/{len(dataloader)}]"
                    f" [D:{loss_D1.item():.4f}]"
                    f" [G:{loss_G.item():.4f}"
                    f" | GAN:{loss_GAN_1.item():.4f}"
                    f" ID:{loss_ID_1.item()+loss_ID_2.item():.4f}"
                    f" Cyc:{loss_cyc_1.item()+loss_cyc_2.item():.4f}]"
                    f" [Enh:{E:.4f}"
                    f" | L1:{loss_enhanced.item():.4f}"
                    f" SSIM:{loss_ssim.item():.4f}"
                    f" Lat:{loss_enhanced_latent.item():.4f}"
                    f" Perc:{loss_perceptual.item():.4f}"
                    f" TV:{loss_tv.item():.4f}]"
                    f" [Atm:{atm_tag}]"
                    f" ETA:{time_left}"
                )

        # ── End of epoch ──────────────────────────────────────────────────────
        if epoch % opt.sample_interval == 0:
            sample_images(epoch)
            print(f"  Snapshot – epoch {epoch}")

        lr_scheduler_G.step()
        lr_scheduler_D1.step()

        # ── FIX F5: Per-epoch validation ──────────────────────────────────────
        if has_val_data and epoch >= 5:  # start validating after epoch 5
            avg_psnr, avg_ssim = validate_epoch(epoch)
            val_history.append((epoch, avg_psnr, avg_ssim))
            is_best = avg_psnr > best_psnr
            if is_best:
                best_psnr = avg_psnr
                best_epoch = epoch
            marker = " ★ BEST" if is_best else ""
            print(f"  Val PSNR: {avg_psnr:.2f} | SSIM: {avg_ssim:.4f}{marker}")

        # ── Save checkpoints ──────────────────────────────────────────────────
        if epoch % opt.checkpoint_interval == 0 or epoch >= 25:
            def _save(name, model):
                torch.save(model.state_dict(),
                           os.path.join(models_dir, f"{name}_{epoch}.pth"))
            for name, model in [("c_Enc", c_Enc), ("G", G),
                                 ("real_sty_Enc", real_sty_Enc),
                                 ("syn_sty_Enc", syn_sty_Enc),
                                 ("T", T), ("D", D),
                                 ("atm_sty_Enc", atm_sty_Enc)]:
                _save(name, model)
            print(f"  Checkpoints saved – epoch {epoch}")

    # ── Print validation summary ──────────────────────────────────────────────
    if val_history:
        print("\n" + "="*60)
        print("VALIDATION SUMMARY")
        print("="*60)
        for ep, p, s in val_history:
            marker = " ★" if ep == best_epoch else ""
            print(f"  Epoch {ep:3d}: PSNR={p:.2f}  SSIM={s:.4f}{marker}")
        print(f"\n  BEST: epoch {best_epoch} with PSNR={best_psnr:.2f}")
        print("="*60)

    return best_epoch, val_history


print("train() v3 defined.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 6 – Testing Functions                                          ║
# ╚══════════════════════════════════════════════════════════════════════╝

def test_REAL_image(epoch, c_Enc, G, real_sty_Enc, T):
    out_path = os.path.join(opt.out_dir, opt.exp_name, 'test_REAL_image', str(epoch))
    os.makedirs(out_path, exist_ok=True)

    transforms_val = [transforms.ToTensor()]
    val_dataloader = DataLoader(
        EnhancedValDataset(transforms_=transforms_val, dataset_path=opt.test_dir),
        batch_size=1, shuffle=False, num_workers=0, pin_memory=True)

    if opt.print_model_complexity:
        num_params = sum(p.numel()
                         for m in [c_Enc, real_sty_Enc, G, T]
                         for p in m.parameters())
        print('Total number of parameters: %d' % num_params)

    time_all = 0
    for i, batch in enumerate(val_dataloader):
        imgReal = batch["img"].to(device)
        name    = batch["name"][0].split(os.sep)[-1]
        with torch.no_grad():
            start        = time.time()
            c_code_Real  = c_Enc(imgReal)
            s_code_Real  = real_sty_Enc(imgReal)
            en_s_code    = T(s_code_Real)
            enhanced     = G(c_code_Real, en_s_code)
            if opt.print_model_complexity and i != 0:
                time_all += time.time() - start
            ndarr = (enhanced.squeeze()
                              .mul(255).add_(0.5).clamp_(0, 255)
                              .permute(1, 2, 0)
                              .to('cpu', torch.uint8).numpy())
            im     = Image.fromarray(ndarr)
            ori_im = Image.open(batch["name"][0])
            im     = im.resize(ori_im.size)
            im.save(os.path.join(out_path, name))

    if len(val_dataloader) > 1:
        print("Total time: %f, average time: %f, FPS: %f, dataloader: %d" % (
            time_all,
            time_all / max(len(val_dataloader) - 1, 1),
            max(len(val_dataloader) - 1, 1) / max(time_all, 1e-9),
            len(val_dataloader)))


def test_SYN_image(c_Enc, G, syn_sty_Enc, T):
    out_path = os.path.join(opt.out_dir, opt.exp_name, 'test_SYN_image', str(opt.checkpoint))
    os.makedirs(out_path, exist_ok=True)

    transforms_val = [transforms.ToTensor()]
    val_dataloader = DataLoader(
        EnhancedValDataset(transforms_=transforms_val, dataset_path=opt.test_dir),
        batch_size=1, shuffle=False, num_workers=0, pin_memory=True)

    if opt.print_model_complexity:
        num_params = sum(p.numel()
                         for m in [c_Enc, syn_sty_Enc, G, T]
                         for p in m.parameters())
        print('Total number of parameters: %d' % num_params)

    time_all = 0
    for i, batch in enumerate(val_dataloader):
        imgSyn = batch["img"].to(device)
        name   = batch["name"][0].split(os.sep)[-1]
        with torch.no_grad():
            start       = time.time()
            c_code_Syn  = c_Enc(imgSyn)
            s_code_Syn  = syn_sty_Enc(imgSyn)
            en_s_code   = T(s_code_Syn)
            enhanced    = G(c_code_Syn, en_s_code)
            if opt.print_model_complexity and i != 0:
                time_all += time.time() - start
            ndarr = (enhanced.squeeze()
                              .mul(255).add_(0.5).clamp_(0, 255)
                              .permute(1, 2, 0)
                              .to('cpu', torch.uint8).numpy())
            im     = Image.fromarray(ndarr)
            ori_im = Image.open(batch["name"][0])
            im     = im.resize(ori_im.size)
            im.save(os.path.join(out_path, name))

    if len(val_dataloader) > 1:
        print("Total time: %f, average time: %f, FPS: %f, dataloader: %d" % (
            time_all,
            time_all / max(len(val_dataloader) - 1, 1),
            max(len(val_dataloader) - 1, 1) / max(time_all, 1e-9),
            len(val_dataloader)))


print("Testing functions defined.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 7 – Run Training  (v3)                                            ║
# ║                                                                          ║
# ║  [PAPER]  = exact value from original paper — do not change             ║
# ║  [NEW]    = atmospheric guidance — tunable                               ║
# ║  [FIXED]  = changed from v2 to fix identified issues                    ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# ── [PATH] Lightning AI paths ─────────────────────────────────────────────────
opt.data_root = '/teamspace/studios/this_studio'
opt.out_dir   = '/teamspace/studios/this_studio/output'
opt.exp_name  = 'uiess_atm_v3'

# ── [PAPER] Architecture (Section IV-A) ──────────────────────────────────────
opt.dim          = 40
opt.style_dim    = 8
opt.n_residual   = 3
opt.n_downsample = 2

# ── [PAPER] Optimiser ─────────────────────────────────────────────────────────
opt.lr    = 5e-4
opt.b1    = 0.5
opt.b2    = 0.999

# ── [PAPER] Schedule ──────────────────────────────────────────────────────────
opt.n_epochs           = 35
opt.epoch              = 0     # set > 0 to resume
opt.decay_epoch        = 20
opt.batch_size         = 1
opt.sample_interval    = 1
opt.checkpoint_interval= 10

# ── [PAPER] Data ──────────────────────────────────────────────────────────────
opt.img_height = 128
opt.img_width  = 128
opt.channels   = 3

# ── [PAPER] Seed ──────────────────────────────────────────────────────────────
opt.seed = 123

# ── [FIXED v3] Atmospheric guidance ───────────────────────────────────────────
#
# lambda_atm = 0.5:
#   L1 loss on style codes: |T(syn_sty_Enc(XB)) - atm_sty_Enc(XC)|.
#   Applied only to en_s_code_B (not both domains).
#   Much smaller than lambda_enhanced_latent=3 → auxiliary guidance only.
#   Paper's Llatent already constrains en_s_code_B ≈ en_s_code_A,
#   so guiding B automatically guides A.
opt.lambda_atm = 0.5

# atm_warmup_epochs = 3:
#   Let the base model stabilize its latent space before adding
#   atmosphere guidance. With 35 epochs total, 3 is a good balance.
opt.atm_warmup_epochs = 3

# ─── Summary ────────────────────────────────────────────────────────────────
print("=" * 68)
print(f"  Experiment      : {opt.exp_name}")
print(f"  Device          : {device}")
print(f"  Epochs          : {opt.n_epochs}  (LR decay @ {opt.decay_epoch})")
print(f"  Base LR         : {opt.lr}")
print(f"  lambda_atm      : {opt.lambda_atm}  (L1, en_s_code_B only)")
print(f"  warmup epochs   : {opt.atm_warmup_epochs}")
print(f"  Grad clip       : NONE  (matches original paper)")
print(f"  Atm loss        : L1  (direction + magnitude)")
print(f"  atm_sty_Enc     : IN optimizer_G  (gradient flows through T)")
print(f"  Validation      : per-epoch PSNR/SSIM on testB")
print("=" * 68)

best_epoch, val_history = train()

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 8 – Run Testing (use best epoch from validation)               ║
# ╚══════════════════════════════════════════════════════════════════════╝

# ── Use best epoch from validation, or manually set ──────────────────────────
try:
    test_epoch = best_epoch
    print(f"Using best epoch from validation: {test_epoch}")
except:
    test_epoch = 29  # fallback
    print(f"Using fallback epoch: {test_epoch}")

# ── Test on SYNTHETIC data (testB) for PSNR/SSIM ─────────────────────────────
opt.test_dir   = '/teamspace/studios/this_studio/testB'
opt.model_dir  = '/teamspace/studios/this_studio/output/saved_models/uiess_atm_v3'
opt.data_root  = '/teamspace/studios/this_studio'
opt.out_dir    = '/teamspace/studios/this_studio/output'
opt.exp_name   = 'uiess_atm_v3'
opt.checkpoint = test_epoch

# ── Build models ──────────────────────────────────────────────────────────────
c_Enc_test        = ContentEncoder(dim=opt.dim, n_downsample=opt.n_downsample,
                              n_residual=opt.n_residual).to(device)
G_test            = Generator(dim=opt.dim, n_upsample=opt.n_downsample,
                         n_residual=opt.n_residual,
                         style_dim=opt.style_dim).to(device)
real_sty_Enc_test = StyleEncoder(dim=opt.dim, n_downsample=opt.n_downsample,
                             style_dim=opt.style_dim).to(device)
syn_sty_Enc_test  = StyleEncoder(dim=opt.dim, n_downsample=opt.n_downsample,
                             style_dim=opt.style_dim).to(device)
T_test            = StyleTransformUnit(dim=opt.dim, style_dim=opt.style_dim).to(device)

# ── Load weights ──────────────────────────────────────────────────────────────
ckpt = opt.checkpoint
c_Enc_test.load_state_dict(
    torch.load(os.path.join(opt.model_dir, "c_Enc_%d.pth" % ckpt), map_location=device))
G_test.load_state_dict(
    torch.load(os.path.join(opt.model_dir, "G_%d.pth" % ckpt), map_location=device))
real_sty_Enc_test.load_state_dict(
    torch.load(os.path.join(opt.model_dir, "real_sty_Enc_%d.pth" % ckpt), map_location=device))
syn_sty_Enc_test.load_state_dict(
    torch.load(os.path.join(opt.model_dir, "syn_sty_Enc_%d.pth" % ckpt), map_location=device))
T_test.load_state_dict(
    torch.load(os.path.join(opt.model_dir, "T_%d.pth" % ckpt), map_location=device))

# Set eval mode
for m in [c_Enc_test, G_test, real_sty_Enc_test, syn_sty_Enc_test, T_test]:
    m.eval()

print(f"Loaded checkpoint epoch {ckpt}. Running: test_SYN_image")
test_SYN_image(c_Enc=c_Enc_test, G=G_test, syn_sty_Enc=syn_sty_Enc_test, T=T_test)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 9 – Compute PSNR/SSIM on synthetic test set                    ║
# ╚══════════════════════════════════════════════════════════════════════╝

import os
import cv2
import numpy as np
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

# paths
gt_dir = "/teamspace/studios/this_studio/testB_label"
pred_dir = f"/teamspace/studios/this_studio/output/uiess_atm_v3/test_SYN_image/{test_epoch}"

gt_files = sorted(os.listdir(gt_dir))
pred_files = sorted(os.listdir(pred_dir))

psnr_list = []
ssim_list = []

for i in range(min(len(gt_files), len(pred_files))):
    gt_path = os.path.join(gt_dir, gt_files[i])
    pred_path = os.path.join(pred_dir, pred_files[i])

    gt = cv2.imread(gt_path)
    pred = cv2.imread(pred_path)

    if gt is None or pred is None:
        continue

    gt = cv2.cvtColor(gt, cv2.COLOR_BGR2RGB)
    pred = cv2.cvtColor(pred, cv2.COLOR_BGR2RGB)

    # Resize pred to match gt if needed
    if pred.shape != gt.shape:
        pred = cv2.resize(pred, (gt.shape[1], gt.shape[0]))

    psnr = peak_signal_noise_ratio(gt, pred, data_range=255)
    s = structural_similarity(gt, pred, channel_axis=2)

    psnr_list.append(psnr)
    ssim_list.append(s)

print("="*60)
print(f"🔥 FINAL RESULTS (v3, Epoch {test_epoch})")
print(f"Average PSNR: {np.mean(psnr_list):.4f}")
print(f"Average SSIM: {np.mean(ssim_list):.4f}")
print("="*60)
print(f"\nComparison:")
print(f"  v2 results: PSNR=20.38  SSIM=0.7822")
print(f"  v3 results: PSNR={np.mean(psnr_list):.2f}  SSIM={np.mean(ssim_list):.4f}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 10 – Test on real images (testA) for UIQM/UCIQE               ║
# ╚══════════════════════════════════════════════════════════════════════╝

# Switch to testA (real images)
opt.test_dir = '/teamspace/studios/this_studio/testA'

print(f"Running test_REAL_image with epoch {test_epoch}")
test_REAL_image(epoch=test_epoch, c_Enc=c_Enc_test, G=G_test,
                real_sty_Enc=real_sty_Enc_test, T=T_test)

print("\nReal image testing complete!")
print("Compute UIQM/UCIQE on the output images in:")
print(f"  {opt.out_dir}/{opt.exp_name}/test_REAL_image/{test_epoch}/")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 11 – Plot Validation History                                    ║
# ╚══════════════════════════════════════════════════════════════════════╝

try:
    if val_history:
        epochs_list = [v[0] for v in val_history]
        psnrs = [v[1] for v in val_history]
        ssims = [v[2] for v in val_history]

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

        ax1.plot(epochs_list, psnrs, 'b-o', markersize=4)
        ax1.axhline(y=20.38, color='r', linestyle='--', label='v2 PSNR (20.38)')
        ax1.axvline(x=best_epoch, color='g', linestyle=':', alpha=0.7, label=f'Best epoch ({best_epoch})')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('PSNR (dB)')
        ax1.set_title('Validation PSNR per Epoch')
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        ax2.plot(epochs_list, ssims, 'g-o', markersize=4)
        ax2.axhline(y=0.7822, color='r', linestyle='--', label='v2 SSIM (0.7822)')
        ax2.axvline(x=best_epoch, color='b', linestyle=':', alpha=0.7, label=f'Best epoch ({best_epoch})')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('SSIM')
        ax2.set_title('Validation SSIM per Epoch')
        ax2.legend()
        ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig(os.path.join(opt.out_dir, f'{opt.exp_name}_val_history.png'), dpi=150)
        plt.show()
        print(f"Plot saved to {opt.out_dir}/{opt.exp_name}_val_history.png")
except:
    print("No validation history available to plot.")